<a href="https://colab.research.google.com/github/sneyx123-github/CopilotStudioSamples/blob/master/DrStop_DnsMapping_v1.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The provided Python notebook efficiently manages Cloud Run
deployments by handling region compatibility, existing state errors, and the 24-hour lock restriction for SSL challenges. A pre-flight DNS check using Python's socket library is recommended to enhance the tool's plug-and-play capability by verifying IP matches before gcloud execution. You can find the sample notebook at GitHub.

---

https://stackoverflow.com/questions/62596466/how-can-i-run-notebooks-of-a-github-project-in-google-colab

---



In [8]:
from google.colab import auth
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"

# Melde dich mit deinem Google-Konto an
auth.authenticate_user()

# Setze das Projekt für die gcloud CLI
!gcloud config set project {PROJECT_ID}


Updated property [core/project].


In [ ]:
import time
from datetime import datetime

# WICHTIG: Nur die Subdomain verwenden, kein http:// oder ://
SERVICE = "gradio-int"
DOMAIN = f"{SERVICE}.us.kommunikator-gmbh.com"
REGION = "europe-west1"

if 1:
  # 1. Altes Mapping löschen
  print(f"🗑️ Lösche altes Mapping für {DOMAIN}...")
  !gcloud beta run domain-mappings delete --domain {DOMAIN} --region {REGION} --quiet

  time.sleep(10)

  # 2. Mapping neu erstellen
  print(f"🚀 Erstelle Mapping neu für {DOMAIN}...")
  !gcloud beta run domain-mappings create --service {SERVICE} --domain {DOMAIN} --region {REGION}

# 3. Überwachungsschleife
print(f"⏳ Überwachung gestartet für {DOMAIN}. Prüfe alle 5 Minuten...")

while True:
    status_output = !gcloud beta run domain-mappings describe --domain {DOMAIN} --region {REGION} --format="yaml(status.conditions)"
    status_str = "\n".join(status_output)
    now = datetime.now().strftime('%H:%M:%S')

    if "status: 'True'\n    type: Ready" in status_str:
        print(f"✅ [{now}] ERFOLG! Deine App ist LIVE unter https://{DOMAIN}")
        break
    elif "message: Certificate issuance pending" in status_str:
        print(f"🔄 [{now}] Google prüft DNS... (Certificate issuance pending)")
    elif "reason: CertificatePending" in status_str:
        print(f"⏳ [{now}] Warte auf Zertifikat-Erstellung...")
    else:
        print(f"📡 [{now}] Status: {status_str[:100]}...") # Zeigt den Anfang des Status an

    time.sleep(300)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"
DOMAIN_SERVICE = "gradio-service.us.kommunikator-gmbh.com"
REGION = "europe-west1"

print(f"Describing domain mapping for {DOMAIN_SERVICE}:")
!gcloud beta run domain-mappings describe --domain {DOMAIN_SERVICE} --region {REGION} --project {PROJECT_ID} --format="yaml"

In [12]:
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"
DOMAIN_TEST = "gradio-int.us.kommunikator-gmbh.com"
REGION = "europe-west1"

print(f"Describing domain mapping for {DOMAIN_TEST}:")
!gcloud beta run domain-mappings describe --domain {DOMAIN_TEST} --region {REGION} --project {PROJECT_ID} --format="yaml"

Describing domain mapping for gradio-int.us.kommunikator-gmbh.com:
ERROR: (gcloud.beta.run.domain-mappings.describe) NOT_FOUND: Resource 'gradio-int.us.kommunikator-gmbh.com' of kind 'DOMAIN_MAPPING' in region 'europe-west1' in project 'project-46810a95-b6e5-47e4-adb' does not exist. This command is authenticated as simonborisney@gmail.com which is the active account specified by the [core/account] property.


In [10]:
!gcloud auth login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=2cyGqa4HShg490OcWpZB2hhjg8Xmvq&prompt=consent&token_usage=remote&access_type=offline&code_challenge=WmYOXB5wfqdCcZGdZaKClVVprTkoaukvAu8DOfilMm4&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E9_FD0NorwiZXkexUnbkWPlIwjSqT9mJ-lTStwOIQChfhuyfXJgnMfUVr441xa28A

You are now logged in as [simonborisney@gmail.com].
Your current projec